In [2]:
import pandas as pd
import pandas_gbq
from google.cloud import bigquery
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from tqdm.auto import tqdm
from google.oauth2 import service_account
import numpy as np

# 1. 신분증(JSON 키) 경로 지정
KEY_PATH = '../google_key.json'

# 2. 인증 객체 생성
credentials = service_account.Credentials.from_service_account_file(KEY_PATH)

# 3. 프로젝트 ID 설정
project_id = 'gdelt-analysis-494301'

# 4. 데이터 불러오기
query = "SELECT SQLDATE FROM `gdelt-bq.full.events` LIMIT 5"
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("인증 성공! 데이터를 가져왔습니다.")

Downloading: 100%|██████████|
인증 성공! 데이터를 가져왔습니다.


In [3]:
# 1. 고위험군 CAMEO 코드 리스트
target_cameo_codes = [
    '150', '151', '152', '153', '154', '155', 
    '190', '191', '192', '193', '194', '195', '196', 
    '200', '201', '202', '203', '204'
]
formatted_codes = ", ".join([f"'{code}'" for code in target_cameo_codes])

query = f"""
SELECT 
    SQLDATE, 
    EventCode,
    GoldsteinScale, 
    NumMentions, 
    AvgTone,
    ActionGeo_Type,
    ActionGeo_Lat, 
    ActionGeo_Long, 
    SOURCEURL
FROM `gdelt-bq.full.events`
WHERE SQLDATE >= 20130401 
  AND (
    (Actor1CountryCode = 'CHN' AND Actor2CountryCode = 'TWN') OR 
    (Actor1CountryCode = 'TWN' AND Actor2CountryCode = 'CHN')
  )
  AND EventCode IN ({formatted_codes})
  AND IsRootEvent = 1                 -- 핵심 사건만 필터링
  AND ActionGeo_Type IN (3, 4, 5)
"""

# 2. 데이터 저장
df = pandas_gbq.read_gbq(query, project_id=project_id, credentials=credentials)

print("데이터 불러오기 완료")
display(df.head())


Downloading: 100%|██████████|
데이터 불러오기 완료


,SQLDATE,EventCode,GoldsteinScale,NumMentions,AvgTone,ActionGeo_Type,ActionGeo_Lat,ActionGeo_Long,SOURCEURL
0,20260510,194,-10.0,16,-2.429197,4,24.9139,118.586,https://www.taipeitimes.com/News/front/archive...
1,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
2,20260510,154,-7.2,1,-3.183521,4,25.0478,121.532,https://news.ltn.com.tw/news/focus/breakingnew...
3,20220320,195,-10.0,12,-1.565588,4,39.9289,116.388,https://news.webindia123.com/news/Articles/Wor...
4,20220320,195,-10.0,6,-1.565588,4,39.9289,116.388,https://news.webindia123.com/news/Articles/Wor...


In [4]:
df['SQLDATE'] = pd.to_datetime(df['SQLDATE'].astype(str), format='%Y%m%d')

df.to_csv("01_data/raw/gdelt_raw.csv", index=False)
print(f"저장 완료: {len(df)}행")

저장 완료: 11937행


In [5]:
import pandas as pd

df = pd.read_csv("01_data/processed/final_priority.csv")

print(df[['ActionGeo_Lat', 'ActionGeo_Long']].describe())
print(f"\n좌표 결측치: {df[['ActionGeo_Lat', 'ActionGeo_Long']].isnull().sum().to_dict()}")
print(f"\n좌표 샘플:\n{df[['ActionGeo_Lat', 'ActionGeo_Long', 'priority_score']].head(10)}")

       ActionGeo_Lat  ActionGeo_Long
count     177.000000      177.000000
mean       31.823555      115.108647
std         9.472259       23.631831
min       -35.283300      -77.036400
25%        25.047800      116.388000
50%        32.526100      116.388000
75%        39.928900      121.532000
max        55.752200      149.217000

좌표 결측치: {'ActionGeo_Lat': 0, 'ActionGeo_Long': 0}

좌표 샘플:
   ActionGeo_Lat  ActionGeo_Long  priority_score
0        24.0000         119.000        0.779269
1        39.9289         116.388        0.717561
2        24.0000         119.000        0.690566
3        25.0478         121.532        0.603822
4        24.0000         119.000        0.603720
5        24.4367         118.318        0.589832
6        28.7925         117.262        0.586239
7        39.9289         116.388        0.573689
8        22.1094         120.874        0.527688
9        39.9289         116.388        0.526684


In [6]:
import pandas as pd
df = pd.read_csv("01_data/processed/spike_events.csv")
print(df.columns.tolist())
print(df.head(3))

['SQLDATE', 'DailyMentions', 'EventCount', 'AvgGoldstein', 'AvgTone', 'MA_7', 'MA_14', 'MA_30', 'MoM_rate', 'is_spike']
      SQLDATE  DailyMentions  EventCount  AvgGoldstein   AvgTone        MA_7  \
0  2018-04-19            457           5         -7.76 -2.213559  214.428571   
1  2019-01-15            868           3        -10.00 -1.978939  224.000000   
2  2019-01-16           1354           3        -10.00 -2.558135  400.285714   

        MA_14  MA_30     MoM_rate  is_spike  
0  344.642857  452.1   315.454545      True  
1  218.500000  299.9   623.333333      True  
2  283.785714  314.0  1028.333333      True  


In [8]:
import pandas as pd
df = pd.read_csv("01_data/processed/final_priority_geo.csv")
print(df.columns.tolist())

['SQLDATE', 'EventCode', 'GoldsteinScale', 'NumMentions', 'AvgTone', 'ActionGeo_Type', 'ActionGeo_Lat', 'ActionGeo_Long', 'SOURCEURL', 'score_mentions', 'score_goldstein', 'score_tone', 'score_geo', 'priority_score', 'geo_level']
